# SQL Multi-Table Query Strategies

**Purpose:** This guide is a quick-reference strategy sheet for SQL interview questions that involve combining data across multiple tables. It covers the most common multi-table patterns you'll encounter — joining for lookups, generating complete combinations, finding mismatches, aggregating across relationships, and more — with a focus on identifying the right join strategy before writing any code, choosing the most efficient approach, and explaining your reasoning clearly to an interviewer.

---

### Table of Contents

**[1. First Step: Identify the Pattern](#1-first-step-identify-the-pattern)** — Master reference table: pattern → join type → method → rationale

**[2. Decision Tree (5-Second Version)](#2-decision-tree-5-second-version)** — Quick mental flowchart for choosing the right join

**[3. Pattern Library with Examples](#3-pattern-library-with-examples)**
- [A. Lookup / Enrich (INNER JOIN)](#pattern-a-inner-join)
- [B. Include All / Preserve Unmatched (LEFT JOIN)](#pattern-b-left-join)
- [C. Complete Combinations (CROSS JOIN)](#pattern-c-cross-join)
- [D. Complete Combinations with Counts (CROSS JOIN + LEFT JOIN)](#pattern-d-cross-join-left-join)
- [E. Find Mismatches / Missing Data (LEFT JOIN + IS NULL)](#pattern-e-find-missing)
- [F. Combine Similar Tables (UNION / UNION ALL)](#pattern-f-union)
- [G. Self-Referencing (Self JOIN)](#pattern-g-self-join)
- [H. Filter by Another Table (EXISTS / IN)](#pattern-h-exists-in)
- [I. Aggregate Then Join (Subquery/CTE + JOIN)](#pattern-i-aggregate-then-join)
- [J. Range JOIN / Non-Equi JOIN (JOIN with BETWEEN)](#pattern-j-range-join)
- [K. Relational Division ("Bought ALL")](#pattern-k-relational-division)
- [L. UNION ALL for Separate Queries & Data Reshaping](#pattern-l-union-all-separate-queries)

**[4. Critical Efficiency Rules](#4-critical-efficiency-rules)** — Five rules to avoid common performance traps

**[5. JOIN Type Quick Reference](#5-join-type-quick-reference)** — Side-by-side comparison of all join types

**[6. Row Count Diagnostics](#6-row-count-diagnostics)** — Verify you picked the right JOIN with QA queries

**[7. How to Think from the Vignette](#7-how-to-think-from-the-vignette)** — Step-by-step approach + worked examples

**[8. Interview Prompt Templates](#8-interview-prompt-templates)** — Ready-made phrases for thinking out loud

**[9. What to Avoid](#9-what-to-avoid)** — Common mistakes and their fixes

**[10. Final Takeaway](#10-final-takeaway)** — The 4 most common patterns to memorize

<hr style="border: 3px solid black;">

<a id='1-first-step-identify-the-pattern'></a>

## 1. First Step: Identify the Pattern

Before writing **any** SQL, ask yourself:

> *How do these tables relate, and what combination of their data does the question need?*

The table below combines **pattern recognition**, **join type**, **function selection**, and **efficiency ranking** into a single reference. Methods are listed from most efficient (1) to least efficient.

| Pattern | Signal Words | Tables Involved | Rank | Method | Rationale |
|---|---|---|---|---|---|
| **Lookup / Enrich** | "get the name", "include details", "add info from" | Fact + Dimension | 1 | `INNER JOIN` | Matches rows on key; only returns matches |
| | | | 2 | Correlated Subquery in `SELECT` | Row-by-row lookup; works but slower on large data |
| **Include All (preserve unmatched)** | "even if no match", "including those with zero", "all students" | Fact + Dimension | 1 | `LEFT JOIN` | Keeps all rows from left table; NULLs for no match |
| | | | 2 | `RIGHT JOIN` | Same logic, opposite direction; less conventional |
| **Complete Combinations** | "every student × every subject", "all possible pairs", "each with each" | Two independent lists | 1 | `CROSS JOIN` | Cartesian product — every row paired with every row |
| | | | 2 | Implicit cross join (comma in `FROM`) | Same result, less readable |
| **Complete Combinations + Counts** | "how many times each X did each Y", "count per combination including zeros" | Two lists + Fact | 1 | `CROSS JOIN` + `LEFT JOIN` + `GROUP BY` | Build all pairs, then attach and count actuals |
| | | | 2 | `CROSS JOIN` + Correlated Subquery | Builds pairs then counts per-row; slower |
| **Find Mismatches / Missing** | "not in", "who didn't", "which have no", "never attended" | Fact + Dimension | 1 | `LEFT JOIN` + `WHERE IS NULL` | Join then filter for non-matches |
| | | | 2 | `NOT EXISTS` | Correlated subquery; readable but can be slower |
| | | | 3 | `NOT IN` | Breaks with NULLs; avoid unless column is NOT NULL |
| **Combine Similar Rows** | "together", "merge", "all from both" | Two tables with same structure | 1 | `UNION ALL` | Stacks rows; keeps duplicates |
| | | | 2 | `UNION` | Stacks rows; removes duplicates (extra cost) |
| **Self-Referencing** | "manager", "parent", "reports to", "hierarchy" | Single table referencing itself | 1 | Self `JOIN` | Table joined to itself on parent/child key |
| | | | 2 | Recursive CTE | For multi-level hierarchy traversal |
| **Compare Across Tables** | "higher than average of other table", "exists in both" | Two related tables | 1 | `EXISTS` / `IN` with subquery | Filter one table based on another |
| | | | 2 | `JOIN` + `WHERE` | Join then filter; may produce duplicates |
| **Aggregate Then Join** | "total per group then combine", "summary + detail" | Fact + Aggregated Fact | 1 | Subquery/CTE + `JOIN` | Pre-aggregate, then join clean result |
| | | | 2 | Window function + `JOIN` | Add aggregate column without collapsing, then join |
| **Range JOIN (Non-Equi)** | "price valid between dates", "join on range", "date overlaps" | Fact + Dimension (range) | 1 | `JOIN` with `BETWEEN` or inequality | Match rows based on range conditions; can expand row count |
| | | | 2 | `JOIN` with `AND` conditions | Same logic, explicit; slightly more verbose |
| **Relational Division (Bought ALL)** | "customers who bought ALL products", "students who passed EVERY exam", "has all items" | Fact + Reference Set | 1 | `GROUP BY` + `HAVING COUNT(DISTINCT) = (SELECT COUNT(*))` | Find entities matching every item in reference set |
| | | | 2 | `GROUP BY` + `HAVING COUNT(*) = (SELECT COUNT(DISTINCT))` | Variant; less robust with duplicates |
| **UNION ALL (Separate Queries or Reshape)** | "Find X. Also find Y.", "combine requester_id and accepter_id", "stack two unrelated results" | Two independent queries or columns | 1 | `UNION ALL` with two `SELECT` blocks | Each query independent; simple vertical stacking |
| | | | 2 | `UNION ALL` with data reshaping | Reshape wide data (multiple columns) to long format (one column) |

<hr style="border: 3px solid black;">

<a id='2-decision-tree-5-second-version'></a>

## 2. Decision Tree (5-Second Version)

Use this quick mental checklist when you first read a multi-table problem:

<div class="fc">
  <div class="fc-node fc-start">READ THE QUESTION<br/>How many tables are involved?<br/>How do they relate?</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">Same structure / similar data?<br/>("combine", "merge", "together")</div>
  <div class="fc-node fc-good">YES → <strong>UNION ALL (or UNION)</strong><br/>Stack rows from both tables<code>SELECT col1, col2 FROM table_a
UNION ALL
SELECT col1, col2 FROM table_b</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Table references itself?<br/>("manager", "parent", "reports")</div>
  <div class="fc-node fc-good">YES → <strong>Self JOIN</strong><br/>(or Recursive CTE for depth)<code>SELECT a.name AS employee,
       b.name AS manager
FROM emp a JOIN emp b
  ON a.manager_id = b.id</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Need ALL combinations?<br/>("every X with every Y")</div>
  <div class="fc-node fc-good">YES → <strong>CROSS JOIN</strong><br/>Then LEFT JOIN for actuals<code>SELECT s.name, sub.name
FROM students s
CROSS JOIN subjects sub</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Need to find what's MISSING?<br/>("never", "didn't", "no match")</div>
  <div class="fc-node fc-good">YES → <strong>LEFT JOIN + WHERE IS NULL</strong><br/>(or NOT EXISTS)<code>SELECT a.* FROM customers a
LEFT JOIN orders b
  ON a.id = b.customer_id
WHERE b.customer_id IS NULL</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Must keep ALL rows from one table even if no match?</div>
  <div class="fc-node fc-good">YES → <strong>LEFT JOIN</strong><br/>+ COALESCE for NULL defaults<code>SELECT a.*, COALESCE(b.val, 0)
FROM main_table a
LEFT JOIN detail b ON a.id = b.fk</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-good">Matching rows only → <strong>INNER JOIN</strong><code>SELECT a.*, b.col
FROM table_a a
JOIN table_b b ON a.id = b.fk</code></div>
</div>


<hr style="border: 3px solid black;">

<a id='3-pattern-library-with-examples'></a>

## 3. Pattern Library with Examples

<a id='pattern-a-inner-join'></a>

### A. Lookup / Enrich (INNER JOIN)

**Signals:** "get the name," "include the department," "find matching records"


<div class="fc" style="max-width: 400px; margin: 20px auto;">
  <div style="display: flex; justify-content: center; align-items: center; gap: 15px;">
    <div style="width: 80px; height: 80px; background: #e8e8e8; border: 2px solid #333; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold;">A</div>
    <div style="font-size: 18px; font-weight: bold;">✓</div>
    <div style="width: 80px; height: 80px; background: #e8e8e8; border: 2px solid #333; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold;">B</div>
  </div>
  <div style="text-align: center; font-size: 0.9em; margin-top: 10px; font-style: italic;">Only the overlap is returned.</div>
</div>

---

**Best approach — `INNER JOIN`**

**Method:** `INNER JOIN` matches rows from two tables on a shared key and returns only the rows that have a match in both tables. Rows without a match in either table are dropped.

**In plain language:** "Give me only the rows where both tables agree on the key — like matching student IDs to get student names."

```sql
-- Get employee names with their department names
SELECT 
    e.employee_id,
    e.employee_name,
    d.department_name
FROM Employees AS e
INNER JOIN Departments AS d
    ON e.department_id = d.department_id;
```

---

**Alternative — Correlated Subquery in SELECT**

**Method:** Instead of joining, you place a subquery in the `SELECT` clause that looks up a value for each row. This runs the subquery once per row in the outer table.

**In plain language:** "For each employee, go look up their department name one at a time."

```sql
SELECT 
    e.employee_id,
    e.employee_name,
    (SELECT d.department_name 
     FROM Departments AS d 
     WHERE d.department_id = e.department_id) AS department_name
FROM Employees AS e;
```

**Why INNER JOIN is better:** The join lets the database match all rows at once using indexes, while the correlated subquery runs a separate lookup for every single row.

<hr style="border: 2px solid black;">

<a id='pattern-b-left-join'></a>

### B. Include All / Preserve Unmatched (LEFT JOIN)

**Signals:** "even if no match," "including zeros," "all customers even those with no orders"


<div class="fc" style="max-width: 450px; margin: 20px auto;">
  <div style="display: flex; justify-content: center; align-items: center; gap: 15px; margin-bottom: 10px;">
    <div style="width: 85px; height: 85px; background: #d4f0d4; border: 2px solid #333; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold; font-size: 0.85em; text-align: center;">A ✓<br><span style="font-size: 0.75em;">kept<br>(NULL)</span></div>
    <div style="font-size: 18px; font-weight: bold;">✓</div>
    <div style="width: 85px; height: 85px; background: #e8e8e8; border: 2px solid #333; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold;">B</div>
  </div>
  <div style="text-align: center; font-size: 0.9em; font-style: italic;">All of A returned; B fills in where matched, NULL where not.</div>
</div>

---

**Best approach — `LEFT JOIN`**

**Method:** `LEFT JOIN` returns all rows from the left table and matching rows from the right table. When there's no match, the right table's columns come back as NULL.

**In plain language:** "Keep every row from my main table, and attach info from the second table where possible — if there's nothing to attach, just put NULL."

```sql
-- All customers with their order count (including those with 0 orders)
SELECT 
    c.customer_id,
    c.customer_name,
    COUNT(o.order_id) AS order_count
FROM Customers AS c
LEFT JOIN Orders AS o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name;
```

---

**Handling NULLs with COALESCE**

**Method:** `COALESCE()` replaces NULL with a default value. Pair it with `LEFT JOIN` to turn NULLs into zeros or meaningful defaults.

**In plain language:** "If there's no match and the value is NULL, show 0 instead."

```sql
SELECT 
    c.customer_id,
    c.customer_name,
    COALESCE(COUNT(o.order_id), 0) AS order_count
FROM Customers AS c
LEFT JOIN Orders AS o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name;
```

<hr style="border: 2px solid black;">

<a id='pattern-c-cross-join'></a>

### C. Complete Combinations (CROSS JOIN)

**Signals:** "every student with every subject," "all possible pairs," "each X combined with each Y"


<div class="fc" style="max-width: 500px; margin: 20px auto;">
  <div style="display: flex; justify-content: center; align-items: center; gap: 20px; margin-bottom: 10px;">
    <div style="background: #e8e8e8; border: 2px solid #333; padding: 15px 10px; text-align: center; min-width: 70px;">A<br><span style="font-size: 0.85em;">4 rows</span></div>
    <div style="font-size: 20px; font-weight: bold;">×</div>
    <div style="background: #e8e8e8; border: 2px solid #333; padding: 15px 10px; text-align: center; min-width: 70px;">B<br><span style="font-size: 0.85em;">3 rows</span></div>
  </div>
  <div style="text-align: center; font-size: 0.9em; font-style: italic;">= 12 row result<br>Every row in A paired with every row in B.</div>
</div>

---

**Best approach — `CROSS JOIN`**

**Method:** `CROSS JOIN` produces the Cartesian product — every row from the first table paired with every row from the second table. If table A has 4 rows and table B has 3 rows, the result has 12 rows.

**In plain language:** "Create every possible combination of items from two lists, like dealing every student a card for every subject."

```sql
-- Generate all student-subject combinations
SELECT 
    s.student_id,
    s.student_name,
    sub.subject_name
FROM Students AS s
CROSS JOIN Subjects AS sub
ORDER BY s.student_id, sub.subject_name;
```

---

**Alternative — Implicit Cross Join (comma syntax)**

**Method:** Listing tables separated by commas in the `FROM` clause without a `WHERE` condition produces the same Cartesian product, but is less readable.

**In plain language:** "Same result, but the intent is hidden — readers might think you forgot a join condition."

```sql
SELECT 
    s.student_id,
    s.student_name,
    sub.subject_name
FROM Students s, Subjects sub
ORDER BY s.student_id, sub.subject_name;
```

**Why explicit CROSS JOIN is better:** It clearly signals to the reader (and interviewer) that the Cartesian product is intentional, not a mistake.

<hr style="border: 2px solid black;">

<a id='pattern-d-cross-join-left-join'></a>

### D. Complete Combinations with Counts (CROSS JOIN + LEFT JOIN)

**Signals:** "how many times each student attended each exam," "count per pair including zeros," "all combinations with totals"


<div class="fc" style="max-width: 600px; margin: 20px auto;">
  <div style="display: flex; justify-content: center; align-items: center; gap: 15px; margin-bottom: 10px; flex-wrap: wrap;">
    <div style="background: #e8e8e8; border: 2px solid #333; padding: 12px 8px; text-align: center; min-width: 60px;">A</div>
    <div style="font-size: 18px; font-weight: bold;">×</div>
    <div style="background: #e8e8e8; border: 2px solid #333; padding: 12px 8px; text-align: center; min-width: 60px;">B</div>
    <div style="font-size: 18px; font-weight: bold;">◄</div>
    <div style="background: #e8e8e8; border: 2px solid #333; padding: 12px 8px; text-align: center; min-width: 70px;">Facts</div>
  </div>
  <div style="text-align: center; font-size: 0.9em; font-style: italic;">All A×B combos; LEFT JOIN attaches counts (0 if missing).</div>
</div>

---

**Best approach — `CROSS JOIN` + `LEFT JOIN` + `GROUP BY`**

**Method:** This is a 3-step pattern: (1) `CROSS JOIN` two dimension tables to build all possible pairs, (2) `LEFT JOIN` to the fact table to attach actual data, (3) aggregate with `COUNT` or `SUM`. This guarantees every combination appears in the output, even those with zero occurrences.

**In plain language:** "First build every possible student-subject pair. Then check the exam records to see which pairs actually happened and how many times. Pairs with no records get a zero."

```sql
-- Count exams per student per subject (including 0)
SELECT 
    s.student_id, 
    s.student_name, 
    sub.subject_name,
    COUNT(e.student_id) AS attended_exams
FROM Students AS s
CROSS JOIN Subjects AS sub
LEFT JOIN Examinations AS e
    ON e.student_id = s.student_id 
    AND e.subject_name = sub.subject_name
GROUP BY s.student_id, s.student_name, sub.subject_name
ORDER BY s.student_id, sub.subject_name;
```

**Key insight:** `COUNT(e.student_id)` counts only non-NULL values. When the `LEFT JOIN` finds no match, `e.student_id` is NULL and the count is 0 — no `CASE WHEN` or `COALESCE` needed.

---

**Alternative — Pre-aggregate then join**

**Method:** First aggregate the fact table in a subquery, then `CROSS JOIN` the dimensions and `LEFT JOIN` the pre-aggregated results. Use `COALESCE` or `CASE WHEN` to handle NULLs.

**In plain language:** "Count the exams first in a separate step, then build all pairs and look up the counts."

```sql
SELECT 
    s.student_id, 
    s.student_name, 
    sub.subject_name,
    COALESCE(e.attended_exams, 0) AS attended_exams
FROM Students AS s
CROSS JOIN Subjects AS sub
LEFT JOIN (
    SELECT student_id, subject_name, COUNT(*) AS attended_exams
    FROM Examinations
    GROUP BY student_id, subject_name
) AS e
    ON e.student_id = s.student_id 
    AND e.subject_name = sub.subject_name
ORDER BY s.student_id, sub.subject_name;
```

**When to use this alternative:** When the fact table is very large and you want to reduce the number of rows before joining. Pre-aggregating shrinks the fact table first, so the `LEFT JOIN` has fewer rows to match.

**Trade-off:** The direct approach (first method) is simpler and usually optimized well by the query planner. The pre-aggregate approach gives you explicit control over performance but adds complexity.

<hr style="border: 2px solid black;">

<a id='pattern-e-find-missing'></a>

### E. Find Mismatches / Missing Data (LEFT JOIN + IS NULL)

**Signals:** "never ordered," "didn't attend," "customers with no," "which have no match"


<div class="fc" style="max-width: 450px; margin: 20px auto;">
  <div style="display: flex; justify-content: center; align-items: center; gap: 15px; margin-bottom: 10px;">
    <div style="width: 85px; height: 85px; background: #d4f0d4; border: 2px solid #333; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold; font-size: 0.8em; text-align: center;">A ✓<br><span style="font-size: 0.7em;">kept<br>only</span></div>
    <div style="font-size: 18px; font-weight: bold;">✗</div>
    <div style="width: 85px; height: 85px; background: #e8e8e8; border: 2px solid #333; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold; font-size: 0.8em; text-align: center;">B<br><span style="font-size: 0.7em;">not<br>used</span></div>
  </div>
  <div style="text-align: center; font-size: 0.9em; font-style: italic;">Only rows in A with NO match in B.</div>
</div>

---

**Best approach — `LEFT JOIN` + `WHERE ... IS NULL`**

**Method:** Join two tables with a `LEFT JOIN`, then filter where the right table's key is NULL. This identifies rows in the left table that have no matching row in the right table.

**In plain language:** "Show me everyone from the first table who has no corresponding entry in the second table."

```sql
-- Find customers who never placed an order
SELECT c.customer_id, c.customer_name
FROM Customers AS c
LEFT JOIN Orders AS o
    ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL;
```

---

**Alternative — `NOT EXISTS`**

**Method:** Use a correlated subquery with `NOT EXISTS` to check for the absence of matching rows.

**In plain language:** "For each customer, check if there's any order — if not, include them."

```sql
SELECT c.customer_id, c.customer_name
FROM Customers AS c
WHERE NOT EXISTS (
    SELECT 1 
    FROM Orders AS o 
    WHERE o.customer_id = c.customer_id
);
```

---

**Avoid — `NOT IN` (with NULLs)**

**Method:** `NOT IN` compares a column against a list from a subquery. If the subquery returns any NULL value, the entire `NOT IN` condition evaluates to UNKNOWN and returns zero rows.

**In plain language:** "This looks simple but breaks silently when NULLs are present — a dangerous trap in interviews."

```sql
-- DANGEROUS: Returns 0 rows if any customer_id in Orders is NULL
SELECT customer_id, customer_name
FROM Customers
WHERE customer_id NOT IN (SELECT customer_id FROM Orders);
```

**Why LEFT JOIN + IS NULL is safest:** It never breaks with NULLs and makes the intent clear.

<hr style="border: 2px solid black;">

<a id='pattern-f-union'></a>

### F. Combine Similar Tables (UNION / UNION ALL)

**Signals:** "combine," "merge," "all records from both," "together"


<div class="fc" style="max-width: 300px; margin: 20px auto;">
  <div style="background: #e8e8e8; border: 2px solid #333; padding: 12px; margin: 8px 0; text-align: center; font-size: 0.9em;">SELECT ... ← Query 1 rows</div>
  <div style="text-align: center; border-top: 2px solid #333; border-bottom: 2px solid #333; padding: 8px; font-weight: bold; background: #f5f5f5;">UNION</div>
  <div style="background: #e8e8e8; border: 2px solid #333; padding: 12px; margin: 8px 0; text-align: center; font-size: 0.9em;">SELECT ... ← Query 2 rows</div>
  <div style="text-align: center; font-size: 0.9em; margin-top: 10px; font-style: italic;">Rows stacked vertically (not side-by-side).</div>
</div>

---

**Best approach — `UNION ALL`**

**Method:** `UNION ALL` stacks the rows from two `SELECT` statements vertically. Both queries must have the same number of columns with compatible data types. It keeps all rows including duplicates.

**In plain language:** "Glue two tables on top of each other — keep every row, even repeats."

```sql
-- Combine active and archived orders
SELECT order_id, customer_id, order_date, 'active' AS source
FROM Active_Orders
UNION ALL
SELECT order_id, customer_id, order_date, 'archived' AS source
FROM Archived_Orders;
```

---

**Alternative — `UNION` (deduplicates)**

**Method:** `UNION` does the same vertical stacking but adds a `DISTINCT` step to remove duplicate rows. This requires sorting the entire result, which is slower.

**In plain language:** "Glue two tables together but remove any rows that appear in both."

```sql
SELECT customer_id FROM Premium_Customers
UNION
SELECT customer_id FROM Newsletter_Subscribers;
```

**When to use UNION vs UNION ALL:**

| Use | When |
|---|---|
| `UNION ALL` | You want all rows, duplicates are fine or impossible |
| `UNION` | You specifically need to eliminate duplicates across both tables |

**Default to `UNION ALL`** — it's faster and you can always add `DISTINCT` later if needed.

<hr style="border: 2px solid black;">

<a id='pattern-g-self-join'></a>

### G. Self-Referencing (Self JOIN)

**Signals:** "manager name," "reports to," "parent category," "compare rows within same table using different roles"


<div class="fc" style="max-width: 300px; margin: 20px auto;">
  <div style="background: #e8e8e8; border: 2px solid #333; padding: 15px; text-align: center;">
    <strong>Table T</strong><br>
    <span style="font-size: 0.85em; display: block; margin-top: 10px;">alias e ──┐<br>alias m ◄─┘<br><em>Same table, two roles</em></span>
  </div>
  <div style="text-align: center; font-size: 0.9em; margin-top: 10px; font-style: italic;">T joined to itself using two aliases.</div>
</div>

---

**Best approach — Self `JOIN`**

**Method:** Join a table to itself using two different aliases. Each alias represents a different "role" of the same table (e.g., employee vs. manager). Connect them on the foreign key that points back to the same table's primary key.

**In plain language:** "Treat the same table as two separate tables — one for the employee, one for the manager — and match them up."

```sql
-- Get each employee with their manager's name
SELECT 
    e.employee_id,
    e.employee_name,
    m.employee_name AS manager_name
FROM Employees AS e
LEFT JOIN Employees AS m
    ON e.manager_id = m.employee_id;
```

---

**Alternative — Recursive CTE (for hierarchy traversal)**

**Method:** A recursive CTE starts with a base case (e.g., top-level managers) and repeatedly joins the table to itself to walk down (or up) the hierarchy level by level.

**In plain language:** "Start at the top of the org chart and keep following the 'reports to' links until you run out of levels."

```sql
-- Get full management chain
WITH RECURSIVE org_chart AS (
    -- Base: top-level managers (no manager)
    SELECT employee_id, employee_name, manager_id, 1 AS level
    FROM Employees
    WHERE manager_id IS NULL
    
    UNION ALL
    
    -- Recursive: employees who report to someone already found
    SELECT e.employee_id, e.employee_name, e.manager_id, oc.level + 1
    FROM Employees AS e
    INNER JOIN org_chart AS oc
        ON e.manager_id = oc.employee_id
)
SELECT * FROM org_chart ORDER BY level, employee_id;
```

**When to use which:**

| Use | When |
|---|---|
| Self JOIN | Single-level lookup (employee → direct manager) |
| Recursive CTE | Multi-level traversal (full hierarchy, depth unknown) |

<hr style="border: 2px solid black;">

<a id='pattern-h-exists-in'></a>

### H. Filter by Another Table (EXISTS / IN)

**Signals:** "who also appear in," "exists in both," "only those with," "filter based on another table"


<div class="fc" style="max-width: 450px; margin: 20px auto;">
  <div style="display: flex; justify-content: center; align-items: center; gap: 15px; margin-bottom: 10px;">
    <div style="width: 85px; height: 85px; background: #d4f0d4; border: 2px solid #333; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold; font-size: 0.8em; text-align: center;">A<br><span style="font-size: 0.7em;">kept<br>if ✓</span></div>
    <div style="font-size: 18px; font-weight: bold;">?</div>
    <div style="width: 85px; height: 85px; background: #e8e8e8; border: 2px solid #333; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold; font-size: 0.8em; text-align: center;">B<br><span style="font-size: 0.7em;">checked<br>only</span></div>
  </div>
  <div style="text-align: center; font-size: 0.9em; font-style: italic;">Rows from A kept only if a match EXISTS in B.</div>
</div>

---

**Best approach — `EXISTS`**

**Method:** `EXISTS` checks whether a correlated subquery returns at least one row. It stops searching as soon as it finds the first match (short-circuits), making it efficient.

**In plain language:** "For each row in my main table, check if there's at least one matching row in the other table."

```sql
-- Find customers who have placed at least one order
SELECT c.customer_id, c.customer_name
FROM Customers AS c
WHERE EXISTS (
    SELECT 1 
    FROM Orders AS o 
    WHERE o.customer_id = c.customer_id
);
```

---

**Alternative — `IN` with subquery**

**Method:** `IN` collects all values from a subquery into a list, then checks if the outer row's value is in that list.

**In plain language:** "Get a list of all customer IDs from Orders, then keep only the customers whose ID is on that list."

```sql
SELECT customer_id, customer_name
FROM Customers
WHERE customer_id IN (SELECT customer_id FROM Orders);
```

---

**Alternative — `INNER JOIN` (but may duplicate)**

**Method:** An `INNER JOIN` can also filter, but if the right table has multiple matching rows, the left table's rows get duplicated.

**In plain language:** "Join and keep only matches — but watch out for duplicates if one customer has multiple orders."

```sql
-- WARNING: may produce duplicate customers
SELECT DISTINCT c.customer_id, c.customer_name
FROM Customers AS c
INNER JOIN Orders AS o
    ON c.customer_id = o.customer_id;
```

**Why EXISTS is preferred:** It never produces duplicates and short-circuits after the first match.

<hr style="border: 2px solid black;">

<a id='pattern-i-aggregate-then-join'></a>

### I. Aggregate Then Join (Subquery/CTE + JOIN)

**Signals:** "total per group then look up," "average by department then show details," "rank then join"


<div class="fc" style="max-width: 500px; margin: 20px auto;">
  <div style="display: flex; justify-content: center; align-items: center; gap: 20px; margin-bottom: 10px;">
    <div style="background: #e8e8e8; border: 2px solid #333; padding: 15px 10px; text-align: center; min-width: 100px;"><strong>Aggregate</strong><br><span style="font-size: 0.85em;">(CTE/Sub)</span></div>
    <div style="font-size: 18px; font-weight: bold;">───JOIN────▶</div>
    <div style="background: #e8e8e8; border: 2px solid #333; padding: 15px 10px; text-align: center; min-width: 100px;"><strong>Detail</strong><br><span style="font-size: 0.85em;">Table</span></div>
  </div>
  <div style="text-align: center; font-size: 0.9em; font-style: italic;">Summarize first, then join the summary to details.</div>
</div>

---

**Best approach — CTE or Subquery + `JOIN`**

**Method:** First aggregate data in a CTE or subquery, then join the aggregated result back to another table. This keeps the logic clean and avoids mixing aggregation with join conditions.

**In plain language:** "Calculate the summary first in a separate step, then combine it with the detail table."

```sql
-- Show each employee with their department's average salary
WITH dept_avg AS (
    SELECT department_id, AVG(salary) AS avg_salary
    FROM Employees
    GROUP BY department_id
)
SELECT 
    e.employee_id,
    e.employee_name,
    e.salary,
    da.avg_salary AS department_avg
FROM Employees AS e
INNER JOIN dept_avg AS da
    ON e.department_id = da.department_id;
```

---

**Alternative — Window Function (no pre-aggregation needed)**

**Method:** A window function can compute the aggregate alongside each row without collapsing the result. No separate subquery or CTE needed.

**In plain language:** "Add the department average as a new column right next to each employee — no grouping step needed."

```sql
SELECT 
    employee_id,
    employee_name,
    salary,
    AVG(salary) OVER (PARTITION BY department_id) AS department_avg
FROM Employees;
```

**When to use which:**

| Use | When |
|---|---|
| CTE/Subquery + JOIN | Need to join aggregated data to a **different** table |
| Window Function | Need aggregate alongside detail rows from the **same** table |

<hr style="border: 2px solid black;">

<a id='pattern-j-range-join'></a>

### J. Range JOIN / Non-Equi JOIN (JOIN with BETWEEN)

**Signals:** "price valid between start_date and end_date," "join on date range," "date overlaps," "price tier"

<div class="fc" style="max-width: 500px; margin: 20px auto;">
  <div style="display: flex; justify-content: center; align-items: center; gap: 20px; margin-bottom: 10px;">
    <div style="background: #e8e8e8; border: 2px solid #333; padding: 15px 10px; text-align: center; min-width: 100px;"><strong>Fact Table</strong><br><span style="font-size: 0.85em;">(Transactions)</span></div>
    <div style="font-size: 18px; font-weight: bold;">📍</div>
    <div style="background: #e8e8e8; border: 2px solid #333; padding: 15px 10px; text-align: center; min-width: 100px;"><strong>Range Table</strong><br><span style="font-size: 0.85em;">(Prices)</span></div>
  </div>
  <div style="text-align: center; font-size: 0.9em; font-style: italic;">Join on BETWEEN or inequality; fact rows match ranges in dimension table.</div>
</div>

---

**Best approach — `JOIN` with `BETWEEN` or inequality**

**Method:** A non-equi join matches rows using a condition other than equality (e.g., `BETWEEN`, `<`, `>`, `>=`, `<=`). Fact table rows are matched to dimension rows based on a range condition.

**In plain language:** "Find the dimension row whose range contains each fact row's value — like finding what price tier a transaction falls into, or what time period a date belongs to."

```sql
-- Average selling price: join transactions to price tiers by date range
-- For each unit sold, find the price that was valid on that purchase_date
SELECT 
    u.unit_id,
    u.units_sold,
    p.price,
    (u.units_sold * p.price) AS total_value
FROM UnitsSold AS u
JOIN Prices AS p
    ON u.product_id = p.product_id
    AND u.purchase_date BETWEEN p.start_date AND p.end_date;
```

---

**Alternative — Explicit `AND` conditions**

**Method:** Use `AND` to combine multiple inequality conditions explicitly. Same logic as `BETWEEN`, just more verbose.

**In plain language:** "Join where the fact value is greater than the lower bound AND less than the upper bound."

```sql
SELECT 
    u.unit_id,
    u.units_sold,
    p.price
FROM UnitsSold AS u
JOIN Prices AS p
    ON u.product_id = p.product_id
    AND u.purchase_date >= p.start_date
    AND u.purchase_date <= p.end_date;
```

---

**Critical Caveat — Row Count Expansion:**

Non-equi joins can produce more output rows than input rows. If a fact row matches multiple ranges (e.g., overlapping date ranges in your dimension table), you'll get one output row per match.

**Verification query:**

```sql
-- Sanity check: do we have unexpected duplicates?
SELECT 
    unit_id,
    COUNT(*) AS match_count
FROM (
    SELECT u.unit_id
    FROM UnitsSold AS u
    JOIN Prices AS p
        ON u.product_id = p.product_id
        AND u.purchase_date BETWEEN p.start_date AND p.end_date
) AS joined
GROUP BY unit_id
HAVING COUNT(*) > 1;
-- If this returns rows, ranges overlap and each fact row matched multiple tiers
```

When to use `BETWEEN` vs explicit `AND`: `BETWEEN` is cleaner for date ranges; use explicit `AND` if ranges are complex or asymmetric (e.g., price >= lower AND price < upper).

<hr style="border: 2px solid black;">

<a id='pattern-k-relational-division'></a>

### K. Relational Division ("Bought ALL")

**Signals:** "customers who bought ALL products," "students who passed EVERY exam," "has all items," "appears in every category"

---

**Best approach — `GROUP BY` + `HAVING COUNT(DISTINCT) = reference count`**

**Method:** Group the fact table by the entity (e.g., customer_id), count the distinct items matched to each entity, then filter for groups whose count equals the total count of items in the reference set.

**In plain language:** "Group by customer, count how many distinct products each bought, and keep only customers whose count matches the total number of products. If there are 5 products total and a customer bought all 5, their count is 5 — they're in the result."

```sql
-- Find customers who bought ALL products
SELECT customer_id
FROM Orders
GROUP BY customer_id
HAVING COUNT(DISTINCT product_id) = (SELECT COUNT(*) FROM Products);
```

**Why this works:**
- If a customer bought products A, B, and C, and there are 3 total products, their `COUNT(DISTINCT product_id)` is 3.
- If they're missing one product, their count is only 2, so they don't match the `HAVING` clause.
- The subquery `(SELECT COUNT(*) FROM Products)` gives the reference count — the total number of items the entity must match.

---

**Alternative — JOIN to reference, then GROUP BY, then check count**

**Method:** Explicitly join to the reference set first, then group and verify the count.

```sql
-- Same logic, more explicit
SELECT o.customer_id
FROM Orders AS o
INNER JOIN Products AS p
    ON 1=1  -- Cartesian join setup (all combinations)
WHERE (o.customer_id, p.product_id) IN (
    SELECT customer_id, product_id FROM Orders
)
GROUP BY o.customer_id
HAVING COUNT(DISTINCT p.product_id) = (SELECT COUNT(*) FROM Products);
```

**Note:** This variant is less common; the original approach (group then count) is cleaner.

---

**Critical Detail — Use COUNT(DISTINCT):**

The `DISTINCT` is crucial if your fact table has duplicate rows (e.g., a customer ordered the same product twice). Without `DISTINCT`, you'd count 2 and think the customer bought 2 different products when they only bought 1.

```sql
-- WRONG: counts 2 for duplicate rows
SELECT customer_id, COUNT(product_id) AS product_count
FROM Orders
GROUP BY customer_id;

-- CORRECT: counts only 1 distinct product even if customer ordered it twice
SELECT customer_id, COUNT(DISTINCT product_id) AS product_count
FROM Orders
GROUP BY customer_id;
```

**Real-world example — Customers Who Bought All Products (#1045):**

| customers | orders | products |
|---|---|---|
| customer_id | order_id, customer_id, product_id | product_id |

```sql
SELECT customer_id
FROM Orders
GROUP BY customer_id
HAVING COUNT(DISTINCT product_id) = (SELECT COUNT(DISTINCT product_id) FROM Orders);
```

Alternative reference: `(SELECT COUNT(*) FROM Products)` if you have a separate products dimension table.

<hr style="border: 2px solid black;">

<a id='pattern-l-union-all-separate-queries'></a>

### L. UNION ALL for Separate Queries & Data Reshaping

**Signals for separate queries:** "Find X. Also find Y." (two independent questions in one output), "return these two different things together"

**Signals for data reshaping:** "combine requester_id and accepter_id columns," "stack multiple columns into one," "transform wide data to long format"

<div class="fc" style="max-width: 400px; margin: 20px auto;">
  <div style="background: #e8e8e8; border: 2px solid #333; padding: 12px; margin: 8px 0; text-align: center; font-size: 0.9em;">Query 1 Result<br/><span style="font-size: 0.85em;">(col_a, col_b)</span></div>
  <div style="text-align: center; border-top: 2px solid #333; border-bottom: 2px solid #333; padding: 8px; font-weight: bold; background: #f5f5f5;">UNION ALL</div>
  <div style="background: #e8e8e8; border: 2px solid #333; padding: 12px; margin: 8px 0; text-align: center; font-size: 0.9em;">Query 2 Result<br/><span style="font-size: 0.85em;">(col_a, col_b)</span></div>
  <div style="text-align: center; font-size: 0.9em; margin-top: 10px; font-style: italic;">Both queries independent; results stacked vertically with same columns.</div>
</div>

---

**Best approach — Two independent `SELECT` blocks with `UNION ALL`**

**Method:** Write each query independently with its own logic, then stack the results vertically. Both queries must return the same number of columns with compatible types.

**In plain language:** "Answer question 1, answer question 2, glue the results on top of each other."

```sql
-- Find user with most ratings + movie with highest avg rating in Feb 2020
-- (Movie Rating #1341 example)

-- First query: user with most ratings
SELECT 
    user_id AS id,
    'user' AS type,
    COUNT(*) AS metric_value
FROM MovieRating
GROUP BY user_id
ORDER BY COUNT(*) DESC, user_id ASC
LIMIT 1

UNION ALL

-- Second query: movie with highest avg rating in Feb 2020
SELECT 
    movie_id AS id,
    'movie' AS type,
    AVG(rating) AS metric_value
FROM MovieRating
WHERE YEAR(created_at) = 2020 AND MONTH(created_at) = 2
GROUP BY movie_id
ORDER BY AVG(rating) DESC, movie_id ASC
LIMIT 1;
```

**Key points:**
- Each query is written from scratch; they don't reference each other.
- The `LIMIT` in each query applies to that query independently, not the final result.
- The column names in the final result come from the first query (or you can rename with `AS` for clarity).

---

**Alternative — UNION ALL for Data Reshaping (Wide → Long)**

**Method:** When a table has multiple similar columns that you want to "unpivot" into one column, use multiple `SELECT` blocks that each pull a different source column.

**In plain language:** "Transform multiple columns into a single column by selecting each column in a separate query, then stacking results."

```sql
-- Friend Requests (#602): combine requester_id and accepter_id into one 'id' column
-- Original table: requester_id, accepter_id, accept_date
-- Goal: single 'id' column with count of acceptances

SELECT 
    requester_id AS id,
    COUNT(*) AS num
FROM FriendRequest
WHERE accept_date IS NOT NULL
GROUP BY requester_id

UNION ALL

SELECT 
    accepter_id AS id,
    COUNT(*) AS num
FROM FriendRequest
WHERE accept_date IS NOT NULL
GROUP BY accepter_id;
```

**Why this approach:**
- Without reshaping, you'd have to write complex conditional logic (e.g., `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`).
- `UNION ALL` keeps the logic simple: each column gets its own SELECT, results stack, then you GROUP BY the consolidated column.

---

**When to use `UNION ALL` vs `UNION`:**

| Use | When |
|---|---|
| `UNION ALL` | Results could have duplicates (expected from logic) OR you don't care about deduplication |
| `UNION` | You specifically need to remove duplicate rows (costs extra computation) |

**Default to `UNION ALL`** — it's faster and simpler.

---

**Common Trap — ORDER BY and LIMIT with UNION:**

- `ORDER BY` and `LIMIT` in subqueries apply **only** to that subquery.
- To order the final result, place `ORDER BY` **after** the entire `UNION ALL` block (without `LIMIT` in subqueries, or with careful consideration).

```sql
-- WRONG: LIMIT 1 in first query returns only 1 user, then stacks with 1 movie
SELECT user_id, 'user' FROM MovieRating ORDER BY rating DESC LIMIT 1
UNION ALL
SELECT movie_id, 'movie' FROM MovieRating ORDER BY rating DESC LIMIT 1;

-- CORRECT (if you want to order final result):
(SELECT user_id AS id, 'user' AS type FROM MovieRating...)
UNION ALL
(SELECT movie_id AS id, 'movie' AS type FROM MovieRating...)
ORDER BY id;
```

<hr style="border: 3px solid black;">

<a id='4-critical-efficiency-rules'></a>

## 4. Critical Efficiency Rules

### Rule 1 — Choose the Right Join Type

| Bad | Good |
|---|---|
| `CROSS JOIN` when you need `INNER JOIN` | Use the most restrictive join that gives correct results |

A `CROSS JOIN` on two 1000-row tables produces 1,000,000 rows. Only use it when you genuinely need all combinations.

### Rule 2 — Pre-aggregate Before Joining Large Tables

| Bad | Good |
|---|---|
| `JOIN` then `GROUP BY` on massive fact table | `GROUP BY` in subquery, then `JOIN` smaller result |

Aggregating first reduces the row count before the join, which is dramatically faster on large tables.

### Rule 3 — Avoid NOT IN with Nullable Columns

| Bad | Good |
|---|---|
| `NOT IN (SELECT nullable_col ...)` | `NOT EXISTS` or `LEFT JOIN + IS NULL` |

If the subquery returns even one NULL, `NOT IN` returns zero rows — silently wrong.

### Rule 4 — Use EXISTS over IN for Large Subqueries

| Bad | Good |
|---|---|
| `IN (SELECT ... FROM huge_table)` | `EXISTS (SELECT 1 FROM huge_table WHERE ...)` |

`EXISTS` short-circuits after the first match. `IN` must materialize the entire subquery result list first.

### Rule 5 — Don't Join When You Can Filter

| Bad | Good |
|---|---|
| `JOIN` + `DISTINCT` to check existence | `EXISTS` or `IN` |

If you only need to check whether a match exists (not retrieve columns from the other table), use `EXISTS` instead of joining and deduplicating.

<hr style="border: 3px solid black;">

<a id='5-join-type-quick-reference'></a>

## 5. JOIN Type Quick Reference

| JOIN Type | What It Returns | NULL Behavior | Use When |
|---|---|---|---|
| `INNER JOIN` | Only matching rows from both tables | No NULLs (unmatched rows dropped) | You only want rows with data in both tables |
| `LEFT JOIN` | All left rows + matching right rows | Right columns are NULL when no match | You need all rows from the primary table |
| `RIGHT JOIN` | All right rows + matching left rows | Left columns are NULL when no match | Rare — rewrite as LEFT JOIN for clarity |
| `FULL OUTER JOIN` | All rows from both tables | NULLs on either side when no match | You need the complete picture from both |
| `CROSS JOIN` | Every row × every row (Cartesian) | No NULLs (no condition to fail) | You need all possible combinations |
| Self `JOIN` | Table joined to itself | Depends on JOIN type used | Table references itself (hierarchy, comparison) |

<hr style="border: 3px solid black;">

<a id='6-row-count-diagnostics'></a>

## 6. Row Count Diagnostics — Verify You Picked the Right JOIN

One of the fastest ways to know if your JOIN is correct is to check whether the **row count makes sense**. A wrong join silently produces too many or too few rows — and wrong row counts mean wrong aggregations.

---

### Expected Row Counts by JOIN Type

| JOIN Type | Expected Row Count | Watch Out For |
|---|---|---|
| `INNER JOIN` | ≤ min(rows_A, rows_B) | Drops unmatched rows — count shrinks if keys don't align |
| `LEFT JOIN` (1:1 key) | = rows_A | Right side fills in or goes NULL — count stays same as left table |
| `LEFT JOIN` (1:many key) | ≥ rows_A | Duplicates left rows when multiple matches exist — count inflates |
| `CROSS JOIN` | = rows_A × rows_B | Cartesian product — count explodes fast |
| `CROSS JOIN` + `LEFT JOIN` + `GROUP BY` | = rows_A × rows_B | After grouping, one row per combination |
| `LEFT JOIN` + `IS NULL` | ≤ rows_A | Only the unmatched rows survive the WHERE filter |
| `UNION ALL` | = rows_query1 + rows_query2 | Stacks everything — no deduplication |
| `UNION` | ≤ rows_query1 + rows_query2 | Deduplicates — count may be less than sum |
| Self `JOIN` | Depends on relationship | Hierarchy: ≤ rows_T; Cartesian self: rows_T × rows_T |

---

### The #1 Diagnostic Rule

> **If your row count after JOIN is larger than expected, you almost certainly have a many-to-many join or duplicate keys.**

This is the most common mistake in SQL interviews. It happens when:

1. **The join key isn't unique on at least one side** — You think it's 1:1 but it's actually 1:many
2. **You forgot to aggregate before joining** — The fact table has multiple rows per key, causing fan-out
3. **You used CROSS JOIN when you meant INNER JOIN** — Every row pairs with every row

---

### QA Queries to Verify Your JOIN

**Step 1 — Check your source table sizes**

```sql
-- Know your starting point
SELECT 'Table_A' AS source, COUNT(*) AS row_count FROM Table_A
UNION ALL
SELECT 'Table_B', COUNT(*) FROM Table_B;
```

**Step 2 — Check for duplicate keys (the root cause of row inflation)**

```sql
-- If any key appears more than once, you have a 1:many or many:many situation
SELECT join_key, COUNT(*) AS occurrences
FROM Table_A
GROUP BY join_key
HAVING COUNT(*) > 1;
```

```sql
-- Check the other table too
SELECT join_key, COUNT(*) AS occurrences
FROM Table_B
GROUP BY join_key
HAVING COUNT(*) > 1;
```

**Step 3 — Compare row count before and after JOIN**

```sql
-- Run this BEFORE aggregating to catch fan-out early
SELECT 
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT a.primary_key) AS unique_left_rows,
    COUNT(DISTINCT b.primary_key) AS unique_right_rows
FROM Table_A AS a
LEFT JOIN Table_B AS b ON a.join_key = b.join_key;
```

**How to read the result:**

| What You See | What It Means |
|---|---|
| `joined_rows` = `unique_left_rows` | Clean 1:1 or 1:0 join — no fan-out |
| `joined_rows` > `unique_left_rows` | Fan-out detected — B has duplicates per key |
| `unique_right_rows` < expected | Some B rows didn't match — check your ON clause |

**Step 4 — Spot-check NULLs after LEFT JOIN**

```sql
-- How many rows from the right table had no match?
SELECT 
    COUNT(*) AS total_rows,
    COUNT(b.primary_key) AS matched_rows,
    COUNT(*) - COUNT(b.primary_key) AS unmatched_rows
FROM Table_A AS a
LEFT JOIN Table_B AS b ON a.join_key = b.join_key;
```

---

### Quick Diagnostic Decision Tree


<div class="fc" style="max-width: 600px;">
  <div class="fc-node fc-start">After your JOIN, check the row count:</div>
  <div class="fc-arrow">▼</div>
  
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-yes">Row count = expected?</div>
      <div class="fc-node fc-good">✓ Proceed with confidence</div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-warn">Row count HIGHER than expected?</div>
      <div style="padding: 10px; border-left: 3px solid #ff9800;">
        <div class="fc-node fc-action">Check for duplicate keys (Step 2)</div>
        <div class="fc-node fc-action" style="margin-top: 8px;">Did you forget to aggregate first? → Use Pattern I</div>
        <div class="fc-node fc-action" style="margin-top: 8px;">Did you accidentally CROSS JOIN? → Add ON clause</div>
      </div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-warn">Row count LOWER than expected?</div>
      <div style="padding: 10px; border-left: 3px solid #ff9800;">
        <div class="fc-node fc-action">INNER JOIN dropping unmatched rows? → Switch to LEFT JOIN</div>
        <div class="fc-node fc-action" style="margin-top: 8px;">WHERE clause filtering too aggressively? → Move filter to ON clause</div>
        <div class="fc-node fc-action" style="margin-top: 8px;">NULL keys not matching? → Check for NULLs in join columns</div>
      </div>
    </div>
  </div>
</div>

---

**Interview tip:** Mention row count verification out loud — "Let me verify: this LEFT JOIN should give me the same number of rows as the customers table, since each customer matches at most one row." This shows the interviewer you think about correctness, not just syntax.
---

### The Aggregate Join Trap: MAX() vs SUM() on Joined Constants

When you JOIN a pre-aggregated subquery result back to the detail table, the aggregated value gets **repeated on every matching row**. This creates a subtle but critical trap.

**Example:** You have 3 rows for "Dog" and the subquery returns `n_poor = 1`. After the LEFT JOIN:

| query_name | rating | n_poor (from subquery) |
|---|---|---|
| Dog | 5 | 1 |
| Dog | 5 | 1 |
| Dog | 1 | 1 |

Now:

| Aggregate | Calculation | Result | Correct? |
|---|---|---|---|
| `MAX(p.n_poor)` | max(1, 1, 1) | **1** | Yes — recovers the constant |
| `SUM(p.n_poor)` | 1 + 1 + 1 | **3** | No — counts the constant once per row |
| `MIN(p.n_poor)` | min(1, 1, 1) | **1** | Yes — also recovers the constant |

**The rule:** After joining an aggregated value, use `MAX()` or `MIN()` to safely recover it. Never use `SUM()` on a value that was already aggregated in a subquery — it will multiply by the number of matching rows.

**The better question to ask:** "Do I even need this subquery?" If the condition can be evaluated row-by-row, use conditional aggregation instead:

```sql
-- Instead of subquery + JOIN + MAX:
ROUND(MAX(p.n_poor::numeric) / COUNT(*) * 100, 2)

-- Just do this directly:
ROUND(AVG(CASE WHEN rating < 3 THEN 100.0 ELSE 0 END), 2)
```

> **Mental model:** After any JOIN, ask: "Did this value get duplicated across rows?" If yes, `SUM()` will overcount — use `MAX()` to recover it, or better yet, eliminate the subquery with conditional aggregation.

<hr style="border: 3px solid black;">

<a id='7-how-to-think-from-the-vignette'></a>

## 7. How to Think from the Vignette

Follow these steps **before writing any SQL**:

### Step 1 — Count the Tables

> *How many tables does the problem give me? What does each one represent?*

Classify each table: **Dimension** (list of entities) vs **Fact** (events/transactions).

### Step 2 — Identify the Relationship

> *How do these tables connect? Is it 1:1, 1:many, or many:many?*

Look at the primary keys and foreign keys. If there's no direct relationship (two independent dimension tables), you likely need a `CROSS JOIN`.

### Step 3 — Check the Output Shape

> *Does the output need every combination? Only matches? All from one side?*

This tells you the join type directly:
- Every combination → `CROSS JOIN`
- Only matches → `INNER JOIN`
- All from one side → `LEFT JOIN`
- What's missing → `LEFT JOIN` + `IS NULL`

### Step 4 — Look for Zero Counts

> *Does the expected output show zeros? That means you need to preserve unmatched rows.*

If you see `0` in the expected output, you definitely need `LEFT JOIN` (not `INNER JOIN`) — and possibly `CROSS JOIN` first to generate the complete combination set.

### Step 5 — Choose Aggregation Strategy

> *Do I aggregate before or after joining?*

- **After joining:** Simpler, works when fact table is small
- **Before joining (subquery):** Better when fact table is large

---

### Worked Examples

**Students × Subjects × Examinations Problem:**

| Step | Answer |
|---|---|
| Count tables | 3 tables: Students (dimension), Subjects (dimension), Examinations (fact) |
| Relationship | Students and Subjects are independent; Examinations links them |
| Output shape | Every student × every subject (all combinations) |
| Zero counts? | Yes — output shows `0` for students who didn't attend |
| Aggregation | COUNT after joining (or pre-aggregate in subquery) |
| **Strategy** | **CROSS JOIN Students × Subjects, LEFT JOIN Examinations, GROUP BY** |

**Customers Without Orders Problem:**

| Step | Answer |
|---|---|
| Count tables | 2 tables: Customers (dimension), Orders (fact) |
| Relationship | 1:many — one customer can have many orders |
| Output shape | Only customers with NO orders |
| Zero counts? | N/A — looking for absence, not counts |
| Aggregation | None needed |
| **Strategy** | **LEFT JOIN + WHERE IS NULL** |

**Employee Manager Problem:**

| Step | Answer |
|---|---|
| Count tables | 1 table: Employees (self-referencing) |
| Relationship | Self-reference — manager_id points to employee_id |
| Output shape | Each employee with their manager name |
| Zero counts? | Use LEFT JOIN to keep employees with no manager (CEO) |
| Aggregation | None needed |
| **Strategy** | **Self LEFT JOIN on manager_id = employee_id** |

<hr style="border: 3px solid black;">

<a id='9-interview-prompt-templates'></a>

## 9. Interview Prompt Templates

Use these phrases during an interview to structure your thinking out loud:

| # | Template | When to Use |
|---|---|---|
| 1 | *"I see [N] tables here — [X] is a dimension table and [Y] is a fact table."* | Table identification |
| 2 | *"The output needs every combination of [X] and [Y], so I'll start with a CROSS JOIN."* | Complete combinations |
| 3 | *"Since the output shows zeros, I need a LEFT JOIN to preserve unmatched rows."* | Zero-count signal |
| 4 | *"I'll pre-aggregate the fact table first to keep the join efficient."* | Large fact tables |
| 5 | *"I'm using LEFT JOIN + WHERE IS NULL because I need to find what's missing."* | Mismatch detection |
| 6 | *"I'm using EXISTS instead of IN because it short-circuits and handles NULLs safely."* | Existence filtering |
| 7 | *"This is a self-referencing problem — the table's manager_id points back to its own employee_id."* | Hierarchy problems |
| 8 | *"I'll use COUNT(e.column) instead of COUNT(*) so NULLs from the LEFT JOIN give me zero."* | Counting with LEFT JOIN |

<hr style="border: 3px solid black;">

<a id='subquery-decision-tree'></a>

## When Do I Need a Subquery Instead of (or With) a JOIN?

JOINs are the default tool for multi-table problems, but some questions require a **subquery** — either instead of a join or combined with one. Use this tree to decide.

### Subquery vs. JOIN Decision Tree


<div class="fc">
  <div class="fc-node fc-start">I need data from another table.<br>Should I JOIN or use a SUBQUERY?</div>
  <div class="fc-arrow">▼</div>
  
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-tag">Do I need COLUMNS from the other table in my output?</div>
      <div style="margin-top: 10px;">
        <div class="fc-label fc-yes">YES</div>
        <div class="fc-node fc-good">Use a JOIN<br>(INNER, LEFT, etc.)</div>
      </div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-start" style="margin-top: 10px;">Do I need a single scalar value from the other table?<br>(count, avg, max, sum)</div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-yes">YES</div>
      <div class="fc-node fc-good">SCALAR SUBQUERY<br>in SELECT or WHERE clause</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-start" style="margin-top: 10px;">Am I checking existence or filtering by a list?</div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-yes">YES</div>
      <div class="fc-node fc-good">EXISTS / IN<br>in WHERE clause<br>"users who bought X"<br>"contests with > 5 entries"</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-good">Subquery in FROM clause<br>(derived table)<br>Pre-agg then join</div>
    </div>
  </div>
</div>

---

### Subquery Placement Guide for Multi-Table Problems

| Put the subquery in… | When you need to… | Signal words | Example |
|---|---|---|---|
| **SELECT** | Inject a global aggregate alongside each row | "percentage of total", "ratio to overall", "compared to all" | `COUNT(x) / (SELECT COUNT(*) FROM Users) * 100` |
| **WHERE** | Filter rows using data from another table without a full join | "users who have…", "only if exists in…", "above the average of…" | `WHERE dept_id IN (SELECT id FROM Dept WHERE region = 'West')` |
| **FROM** | Pre-aggregate or reshape data before joining it | "average per group, then join", "rank within a subset" | `FROM (SELECT dept_id, AVG(salary) FROM Emp GROUP BY dept_id) AS t` |

> **Rule of thumb:** If the other table provides a **single number** → subquery in SELECT or WHERE. If you need **multiple columns** from it → JOIN. If you need to **pre-aggregate** before combining → subquery in FROM (or CTE).

---

### Worked Example: Percentage of Total (SELECT subquery)

**Problem:** Find the percentage of total users registered in each contest.

```sql
SELECT r.contest_id,
       ROUND(
           (COUNT(DISTINCT r.user_id)::numeric
            / (SELECT COUNT(DISTINCT user_id) FROM Users)
           ) * 100, 2
       ) AS percentage
FROM Register AS r
GROUP BY r.contest_id
ORDER BY percentage DESC, r.contest_id ASC;
```

**Why a subquery instead of a JOIN?** The Users table provides a single scalar value (total user count) — we don't need any of its columns in the output. A JOIN would be overkill and could complicate the GROUP BY.

**Where does it go?** In the **SELECT** clause — the scalar total is used in every row's percentage formula as the denominator.


<hr style="border: 3px solid black;">

<a id='10-what-to-avoid'></a>

## 10. What to Avoid

| Bad Pattern | Why It's Bad | Better Alternative |
|---|---|---|
| `NOT IN` with nullable column | Returns 0 rows silently when NULLs present | `NOT EXISTS` or `LEFT JOIN + IS NULL` |
| `CROSS JOIN` when `INNER JOIN` suffices | Produces massive Cartesian product unnecessarily | Use `INNER JOIN` with proper ON condition |
| `JOIN` + `DISTINCT` to check existence | Joins all rows then deduplicates — wasteful | `EXISTS` (short-circuits) |
| Aggregating after joining a large fact table | Join explodes row count before aggregation | Pre-aggregate in subquery, then join |
| Using `RIGHT JOIN` | Confusing — readers have to reverse the logic mentally | Rewrite as `LEFT JOIN` by swapping table order |
| Forgetting `COUNT(column)` vs `COUNT(*)` | `COUNT(*)` counts NULLs; gives wrong totals after LEFT JOIN | `COUNT(specific_column)` skips NULLs |
| Implicit cross join (comma in FROM) | Looks like a missing join condition — causes confusion | Explicit `CROSS JOIN` keyword |

<hr style="border: 3px solid black;">

<a id='11-final-takeaway'></a>

## 11. Final Takeaway

The entire game for multi-table problems is:

```
Count the tables  →  Classify them  →  Pick the join  →  Handle NULLs
```

### Interview Script (Memorize This)

> *"First I count the tables and classify each as dimension or fact. Then I check whether the output needs all combinations, only matches, or missing rows — this tells me the join type. If I see zeros in the expected output, I know I need LEFT JOIN to preserve unmatched rows. Finally, I choose whether to aggregate before or after joining based on the table sizes."*

### The 4 Most Common Multi-Table Patterns

| # | Pattern | Join Strategy |
|---|---|---|
| 1 | Lookup a value from another table | `INNER JOIN` |
| 2 | Keep all rows + attach optional data | `LEFT JOIN` + `COALESCE` |
| 3 | All combinations + counts with zeros | `CROSS JOIN` + `LEFT JOIN` + `COUNT(col)` |
| 4 | Find what's missing | `LEFT JOIN` + `WHERE IS NULL` |

If you can recognize these four patterns instantly, you can solve the vast majority of multi-table SQL interview questions.